# ST3 — Catalogue Intelligence

## Analyse produits, saisonnalité et pricing dynamique

Ce notebook correspond au sous-thème **ST3** du projet e-commerce.

L'objectif est simple : comprendre quels produits génèrent le plus de valeur, quels produits sont saisonniers, quels produits doivent être mis en avant, et quels produits peuvent faire l'objet d'une action de pricing.

À la fin, le notebook exporte des fichiers propres dans `data/gold/` pour préparer le dashboard.

## 1. Objectifs métier du notebook

Dans ce notebook, je cherche à répondre à plusieurs questions concrètes :

1. Quels sont les produits qui génèrent le plus de chiffre d'affaires ?  
2. Est-ce que le chiffre d'affaires est concentré sur une petite partie du catalogue ?  
3. Quels produits sont stratégiques selon la matrice BCG ?  
4. Quels mois ou semaines sont les plus forts en ventes ?  
5. Quels produits sont sensibles au prix ?  
6. Quelles actions pricing peut-on recommander ?


## 2. Import des librairies

On importe les librairies nécessaires pour manipuler les données, créer les graphiques et exporter les résultats.

In [2]:

# Manipulation des données
import pandas as pd
import numpy as np

# Visualisation
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Chemins de fichiers
from pathlib import Path

# Affichage propre dans le notebook
from IPython.display import display, Markdown

# Options d'affichage
pd.set_option("display.max_columns", 80)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

## 3. Détection automatique du dossier projet
Le code ci-dessous retrouve automatiquement la racine du projet, puis prépare les chemins vers :

- `data/silver/` : données propres en entrée ;
- `data/gold/` : fichiers exportés pour Power BI / Tableau.

In [19]:

def find_project_root(start_path: Path = Path.cwd()) -> Path:
    """Retourne le dossier racine du projet en cherchant data/silver."""
    start_path = start_path.resolve()
    candidates = [start_path] + list(start_path.parents)

    for candidate in candidates:
        if (candidate / "data" / "silver").exists():
            return candidate

    raise FileNotFoundError(
        "Impossible de trouver le dossier data/silver. "
        "Place ce notebook dans le projet ou dans le dossier notebooks/."
    )

PROJECT_ROOT = find_project_root()
SILVER_DIR = PROJECT_ROOT / "data" / "silver"
GOLD_DIR = PROJECT_ROOT / "data" / "gold"

GOLD_DIR.mkdir(parents=True, exist_ok=True)

print("Racine projet :", PROJECT_ROOT)
print("Dossier Silver :", SILVER_DIR)
print("Dossier Gold :", GOLD_DIR)

Racine projet : C:\Users\Olfa\OneDrive\Bureau\OneDrive\Documents\Downloads\ecommerce-analytics-project
Dossier Silver : C:\Users\Olfa\OneDrive\Bureau\OneDrive\Documents\Downloads\ecommerce-analytics-project\data\silver
Dossier Gold : C:\Users\Olfa\OneDrive\Bureau\OneDrive\Documents\Downloads\ecommerce-analytics-project\data\gold


## 4. Chargement des datasets ST3

Pour ST3, on utilise principalement :

- `online_retail_full.csv` : ventes, produits, quantités, prix, dates, pays ;
- `online_retail_returns.csv` : retours clients, utiles pour calculer le taux de retour produit.


In [20]:

def first_existing_file(folder: Path, filenames: list[str]) -> Path:
    """Retourne le premier fichier existant dans une liste de noms possibles."""
    for name in filenames:
        path = folder / name
        if path.exists():
            return path
    raise FileNotFoundError(f"Aucun fichier trouvé parmi : {filenames}")

sales_file = first_existing_file(
    SILVER_DIR,
    ["online_retail_full.csv", "online_retail_clean.csv", "data_cleaned.csv"]
)

returns_file = first_existing_file(
    SILVER_DIR,
    ["online_retail_returns.csv"]
)

print("Fichier ventes utilisé :", sales_file.name)
print("Fichier retours utilisé :", returns_file.name)

# Lecture des fichiers
sales = pd.read_csv(
    sales_file,
    dtype={"InvoiceNo": "str", "StockCode": "str", "Description": "str", "Country": "str"},
    low_memory=False
)

returns = pd.read_csv(
    returns_file,
    dtype={"InvoiceNo": "str", "StockCode": "str", "Description": "str", "Country": "str"},
    low_memory=False
)

print("Ventes :", sales.shape)
print("Retours :", returns.shape)

display(sales.head())

Fichier ventes utilisé : online_retail_full.csv
Fichier retours utilisé : online_retail_returns.csv
Ventes : (523281, 9)
Retours : (8872, 9)


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,TotalRevenue
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6.00,2010-12-01 08:26:00,2.55,"17,850.00",United Kingdom,15.30
1,536365,71053,WHITE METAL LANTERN,6.00,2010-12-01 08:26:00,3.39,"17,850.00",United Kingdom,20.34
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8.00,2010-12-01 08:26:00,2.75,"17,850.00",United Kingdom,22.00
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6.00,2010-12-01 08:26:00,3.39,"17,850.00",United Kingdom,20.34
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6.00,2010-12-01 08:26:00,3.39,"17,850.00",United Kingdom,20.34


## 5. Préparation des données

On garde uniquement les lignes de ventes positives pour l'analyse catalogue.

In [ ]:

# Conversion des dates
sales["InvoiceDate"] = pd.to_datetime(sales["InvoiceDate"], errors="coerce")
returns["InvoiceDate"] = pd.to_datetime(returns["InvoiceDate"], errors="coerce")

# Conversion des colonnes numériques
for df in [sales, returns]:
    for col in ["Quantity", "UnitPrice", "CustomerID"]:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

# Harmonisation du chiffre d'affaires
if "TotalRevenue" not in sales.columns:
    if "TotalPrice" in sales.columns:
        sales["TotalRevenue"] = sales["TotalPrice"]
    else:
        sales["TotalRevenue"] = sales["Quantity"] * sales["UnitPrice"]

if "TotalRevenue" not in returns.columns:
    if "TotalPrice" in returns.columns:
        returns["TotalRevenue"] = returns["TotalPrice"]
    else:
        returns["TotalRevenue"] = returns["Quantity"] * returns["UnitPrice"]

sales["TotalRevenue"] = pd.to_numeric(sales["TotalRevenue"], errors="coerce")
returns["TotalRevenue"] = pd.to_numeric(returns["TotalRevenue"], errors="coerce")

# Codes non produits à exclure de l'analyse catalogue
SPECIAL_CODES = {"POST", "D", "C2", "M", "BANK CHARGES", "DOT", "AMAZONFEE", "CRUK"}

sales_st3 = sales.copy()

# On garde les vraies ventes : quantité, prix et CA positifs
sales_st3 = sales_st3[
    (sales_st3["Quantity"] > 0) &
    (sales_st3["UnitPrice"] > 0) &

    (sales_st3["TotalRevenue"] > 0)
].copy()

# Exclusion des codes techniques / frais / remises
sales_st3 = sales_st3[~sales_st3["StockCode"].astype(str).str.upper().isin(SPECIAL_CODES)].copy()

# Suppression des lignes sans informations essentielles
sales_st3 = sales_st3.dropna(subset=["StockCode", "Description", "InvoiceDate"])

print("Lignes initiales ventes :", len(sales))
print("Lignes conservées pour ST3 :", len(sales_st3))
print("Produits uniques conservés :", sales_st3["StockCode"].nunique())

Lignes initiales ventes : 523281
Lignes conservées pour ST3 : 522574
Produits uniques conservés : 3915


## 6. Contrôle rapide des données

Cette étape permet de vérifier la période couverte, le nombre de produits et le volume de ventes analysé.

In [22]:

overview = pd.DataFrame({
    "Indicateur": [
        "Date début",
        "Date fin",
        "Nombre de lignes ventes ST3",
        "Nombre de produits uniques",
        "Nombre de commandes",
        "Nombre de clients identifiés",
        "Pays couverts"
    ],
    "Valeur": [
        sales_st3["InvoiceDate"].min(),
        sales_st3["InvoiceDate"].max(),
        len(sales_st3),
        sales_st3["StockCode"].nunique(),
        sales_st3["InvoiceNo"].nunique(),
        sales_st3["CustomerID"].nunique(),
        sales_st3["Country"].nunique()
    ]
})

display(overview)

,Indicateur,Valeur
0,Date début,2010-12-01 08:26:00
1,Date fin,2011-12-09 12:50:00
2,Nombre de lignes ventes ST3,522574
3,Nombre de produits uniques,3915
4,Nombre de commandes,19776
5,Nombre de clients identifiés,4334
6,Pays couverts,38


## 7. KPIs catalogue

On calcule les premiers indicateurs clés du catalogue : chiffre d'affaires, quantité vendue, nombre de commandes, prix moyen et panier moyen.

In [23]:

total_revenue = sales_st3["TotalRevenue"].sum()
total_quantity = sales_st3["Quantity"].sum()
nb_skus = sales_st3["StockCode"].nunique()
nb_orders = sales_st3["InvoiceNo"].nunique()
avg_unit_price = sales_st3["UnitPrice"].mean()
avg_order_value = sales_st3.groupby("InvoiceNo")["TotalRevenue"].sum().mean()

kpis = pd.DataFrame({
    "KPI": [
        "Chiffre d'affaires total",
        "Quantité vendue",
        "Nombre de SKUs",
        "Nombre de commandes",
        "Prix unitaire moyen",
        "Panier moyen"
    ],
    "Valeur": [
        total_revenue,
        total_quantity,
        nb_skus,
        nb_orders,
        avg_unit_price,
        avg_order_value
    ]
})

display(kpis)

,KPI,Valeur
0,Chiffre d'affaires total,"7,249,630.44"
1,Quantité vendue,"3,695,313.50"
2,Nombre de SKUs,"3,915.00"
3,Nombre de commandes,"19,776.00"
4,Prix unitaire moyen,2.94
5,Panier moyen,366.59


### Lecture métier

Ces KPIs donnent une première vision du catalogue.  
Ils permettent de savoir si l'analyse porte sur un volume suffisant et de préparer les indicateurs utilisés dans le dashboard.

## 8. Construction de la table de performance produit

On crée une table à la maille **1 ligne = 1 produit / SKU**.

Pour chaque produit, on calcule :

- quantité vendue ;
- chiffre d'affaires total ;
- nombre de commandes ;
- nombre de clients ;
- prix moyen ;
- quantité retournée ;
- taux de retour ;
- part du chiffre d'affaires.

In [ ]:

# Agrégation des ventes par produit
product_sales = sales_st3.groupby("StockCode", as_index=False).agg(
    quantity_sold=("Quantity", "sum"),
    total_revenue=("TotalRevenue", "sum"),
    nb_orders=("InvoiceNo", "nunique"),
    nb_customers=("CustomerID", "nunique"),
    avg_unit_price=("UnitPrice", "mean"),
    first_sale=("InvoiceDate", "min"),
    last_sale=("InvoiceDate", "max")
)

# Description produit : on garde la dernière description disponible pour chaque SKU
product_desc = (
    sales_st3.sort_values("InvoiceDate")
    .drop_duplicates("StockCode", keep="last")[["StockCode", "Description"]]
    .rename(columns={"Description": "description"})
)

product_perf = product_sales.merge(product_desc, on="StockCode", how="left")

# Agrégation des retours par produit
returns_st3 = returns.copy()
returns_st3 = returns_st3[~returns_st3["StockCode"].astype(str).str.upper().isin(SPECIAL_CODES)].copy()

returns_agg = returns_st3.groupby("StockCode", as_index=False).agg(
    returned_qty=("Quantity", lambda x: x.abs().sum()),
    return_value=("TotalRevenue", lambda x: x.abs().sum()),
    nb_return_invoices=("InvoiceNo", "nunique")
)

# Fusion ventes + retours
product_perf = product_perf.merge(returns_agg, on="StockCode", how="left")

# Remplacer les retours manquants par 0
for col in ["returned_qty", "return_value", "nb_return_invoices"]:
    product_perf[col] = product_perf[col].fillna(0)

# Taux de retour en quantité
product_perf["return_rate_qty"] = (
    product_perf["returned_qty"] / product_perf["quantity_sold"]
).replace([np.inf, -np.inf], np.nan).fillna(0)

# Part de CA du produit dans le catalogue
product_perf["market_share_pct"] = product_perf["total_revenue"] / product_perf["total_revenue"].sum() * 100

# Tri par chiffre d'affaires
product_perf = product_perf.sort_values("total_revenue", ascending=False).reset_index(drop=True)
product_perf["rank_regit venue"] = np.arange(1, len(product_perf) + 1)

# Affichage des premiers produits
cols_preview = [
    "StockCode", "description", "quantity_sold", "total_revenue", "nb_orders",
    "avg_unit_price", "returned_qty", "return_rate_qty", "market_share_pct"
]

display(product_perf[cols_preview].head(10))

,StockCode,description,quantity_sold,total_revenue,nb_orders,avg_unit_price,returned_qty,return_rate_qty,market_share_pct
0,22423,REGENCY CAKESTAND 3 TIER,"11,282.50","95,134.62",1988,8.44,855.00,0.08,1.31
1,47566,PARTY BUNTING,"12,613.00","66,027.24",1685,5.45,268.00,0.02,0.91
2,85123A,CREAM HANGING HEART T-LIGHT HOLDER,"21,629.50","61,307.19",2198,3.12,"2,578.00",0.12,0.85
3,85099B,JUMBO BAG RED RETROSPOT,"22,390.00","47,772.21",2089,2.49,"1,115.00",0.05,0.66
4,22086,PAPER CHAIN KIT 50'S CHRISTMAS,"11,493.50","37,507.46",1160,3.36,453.00,0.04,0.52
5,84879,ASSORTED COLOUR BIRD ORNAMENT,"21,599.00","36,728.65",1455,1.72,48.00,0.00,0.51
6,23298,SPOTTY BUNTING,"6,682.50","34,205.38",1146,5.28,110.00,0.02,0.47
7,79321,CHILLI LIGHTS,"6,250.50","33,013.13",661,6.01,80.00,0.01,0.46
8,22960,JAM MAKING SET WITH JARS,"7,114.00","30,796.14",1132,5.08,246.00,0.03,0.42
9,22720,SET OF 3 CAKE TINS PANTRY DESIGN,"5,638.50","28,418.85",1385,5.47,153.00,0.03,0.39


## 9. Top 20 produits par chiffre d'affaires

Ce graphique permet d'identifier rapidement les produits qui contribuent le plus au chiffre d'affaires.

Ces produits doivent être surveillés en priorité : stock, prix, disponibilité, qualité et retours.

In [25]:

top20_skus = product_perf.head(20).copy()
top20_skus["description_short"] = top20_skus["description"].str.slice(0, 55)

fig = px.bar(
    top20_skus.sort_values("total_revenue", ascending=True),
    x="total_revenue",
    y="description_short",
    orientation="h",
    title="Top 20 des produits par chiffre d'affaires",
    labels={"total_revenue": "Chiffre d'affaires", "description_short": "Produit"},
    hover_data=["StockCode", "quantity_sold", "nb_orders", "avg_unit_price", "return_rate_qty"]
)

fig.update_layout(height=650)
fig.show()

## 10. Analyse ABC / Pareto

L'analyse ABC sert à classer les produits selon leur contribution au chiffre d'affaires.

- **Classe A** : produits les plus importants, environ 80 % du CA ;
- **Classe B** : produits intermédiaires, environ 15 % du CA ;
- **Classe C** : produits à faible contribution, environ 5 % du CA.

Cette analyse aide à prioriser les efforts commerciaux et marketing.

In [27]:

# Calcul du CA cumulé
product_perf["cumulative_revenue"] = product_perf["total_revenue"].cumsum()
product_perf["cumulative_revenue_pct"] = (
    product_perf["cumulative_revenue"] / product_perf["total_revenue"].sum() * 100
)

# Affectation des classes ABC
product_perf["abc_class"] = np.select(
    [
        product_perf["cumulative_revenue_pct"] <= 80,
        product_perf["cumulative_revenue_pct"] <= 95
    ],
    ["A", "B"],
    default="C"
)

abc_summary = product_perf.groupby("abc_class", as_index=False).agg(
    nb_skus=("StockCode", "nunique"),
    total_revenue=("total_revenue", "sum"),
    quantity_sold=("quantity_sold", "sum")
)

abc_summary["revenue_share_pct"] = abc_summary["total_revenue"] / abc_summary["total_revenue"].sum() * 100
abc_summary["sku_share_pct"] = abc_summary["nb_skus"] / abc_summary["nb_skus"].sum() * 100

# Ordre logique A, B, C
abc_summary["abc_class"] = pd.Categorical(abc_summary["abc_class"], categories=["A", "B", "C"], ordered=True)
abc_summary = abc_summary.sort_values("abc_class")

display(abc_summary)

,abc_class,nb_skus,total_revenue,quantity_sold,revenue_share_pct,sku_share_pct
0,A,921,"5,798,552.66","2,468,568.00",79.98,23.52
1,B,994,"1,088,544.85","903,354.00",15.02,25.39
2,C,2000,"362,532.93","323,391.50",5.00,51.09


In [28]:

# Courbe Pareto sur les 300 premiers produits pour garder un graphique lisible
pareto_view = product_perf.head(300).copy()

fig = px.line(
    pareto_view,
    x="rank_revenue",
    y="cumulative_revenue_pct",
    title="Courbe Pareto — CA cumulé par produits",
    labels={"rank_revenue": "Rang du produit", "cumulative_revenue_pct": "CA cumulé (%)"},
    hover_data=["StockCode", "description", "total_revenue", "abc_class"]
)

fig.add_hline(y=80, line_dash="dash", annotation_text="Seuil 80 %")
fig.add_hline(y=95, line_dash="dash", annotation_text="Seuil 95 %")
fig.update_layout(height=500)
fig.show()

## 11. Matrice BCG du catalogue

La matrice BCG permet de classer les produits selon deux dimensions :

- **part de chiffre d'affaires** : poids du produit dans le catalogue ;
- **croissance** : évolution du CA entre la première et la deuxième moitié de la période.

Les quadrants sont :

- **Étoile** : produit fort et en croissance ;
- **Vache à lait** : produit fort mais croissance plus faible ;
- **Dilemme** : produit encore faible mais en croissance ;
- **Poids mort** : produit faible et peu dynamique.

In [30]:

# Découpage de la période en deux parties
min_date = sales_st3["InvoiceDate"].min()
max_date = sales_st3["InvoiceDate"].max()
split_date = min_date + (max_date - min_date) / 2

sales_st3["period_bcg"] = np.where(
    sales_st3["InvoiceDate"] <= split_date,
    "first_period",
    "second_period"
)

# CA par produit et par période
bcg_pivot = (
    sales_st3.groupby(["StockCode", "period_bcg"])["TotalRevenue"]
    .sum()
    .unstack(fill_value=0)
    .reset_index()
)

for col in ["first_period", "second_period"]:
    if col not in bcg_pivot.columns:
        bcg_pivot[col] = 0

# Croissance du CA entre les deux périodes
bcg_pivot["growth_pct"] = np.where(
    bcg_pivot["first_period"] > 0,
    (bcg_pivot["second_period"] - bcg_pivot["first_period"]) / bcg_pivot["first_period"] * 100,
    np.nan
)

product_perf = product_perf.merge(
    bcg_pivot[["StockCode", "first_period", "second_period", "growth_pct"]],
    on="StockCode",
    how="left"
)

# Seuils de classification
share_threshold = product_perf["market_share_pct"].mean()
growth_threshold = product_perf["growth_pct"].replace([np.inf, -np.inf], np.nan).median()

# Affectation des quadrants
conditions = [
    (product_perf["market_share_pct"] >= share_threshold) & (product_perf["growth_pct"] >= growth_threshold),
    (product_perf["market_share_pct"] >= share_threshold) & ~(product_perf["growth_pct"] >= growth_threshold),
    (product_perf["market_share_pct"] < share_threshold) & (product_perf["growth_pct"] >= growth_threshold),
]

choices = ["Étoile", "Vache à lait", "Dilemme"]
product_perf["bcg_quadrant"] = np.select(conditions, choices, default="Poids mort")

bcg_summary = product_perf.groupby("bcg_quadrant", as_index=False).agg(
    nb_skus=("StockCode", "nunique"),
    total_revenue=("total_revenue", "sum"),
    avg_growth_pct=("growth_pct", "mean")
)

bcg_summary["revenue_share_pct"] = bcg_summary["total_revenue"] / product_perf["total_revenue"].sum() * 100

display(bcg_summary.sort_values("total_revenue", ascending=False))

,bcg_quadrant,nb_skus,total_revenue,avg_growth_pct,revenue_share_pct
3,Étoile,513,"3,674,207.40",777.76,50.68
2,Vache à lait,417,"2,141,173.14",-27.18,29.53
1,Poids mort,1798,"756,572.64",-62.80,10.44
0,Dilemme,1187,"677,677.27",403.12,9.35


In [31]:

# Pour éviter que quelques outliers écrasent le graphique, on affiche la croissance bornée.
bcg_plot = product_perf.sort_values("total_revenue", ascending=False).head(600).copy()
bcg_plot["growth_pct_plot"] = bcg_plot["growth_pct"].clip(lower=-100, upper=500)
bcg_plot["description_short"] = bcg_plot["description"].str.slice(0, 60)

fig = px.scatter(
    bcg_plot,
    x="market_share_pct",
    y="growth_pct_plot",
    size="total_revenue",
    color="bcg_quadrant",
    hover_name="description_short",
    hover_data=["StockCode", "abc_class", "total_revenue", "growth_pct", "market_share_pct"],
    title="Matrice BCG — Produits par part de CA et croissance",
    labels={
        "market_share_pct": "Part du CA (%)",
        "growth_pct_plot": "Croissance du CA (%)",
        "bcg_quadrant": "Quadrant BCG"
    }
)

fig.add_vline(x=share_threshold, line_dash="dash", annotation_text="Seuil part CA")
fig.add_hline(y=growth_threshold, line_dash="dash", annotation_text="Seuil croissance")
fig.update_layout(height=650)
fig.show()

In [32]:

star_count = int((product_perf["bcg_quadrant"] == "Étoile").sum())
cashcow_count = int((product_perf["bcg_quadrant"] == "Vache à lait").sum())

message = f"""
### Lecture métier — BCG

La matrice BCG permet de distinguer les produits à fort potentiel des produits à faible contribution.  
Dans cette analyse, on identifie **{star_count} produits Étoiles** et **{cashcow_count} produits Vaches à lait**.

- Les **Étoiles** doivent être mises en avant et sécurisées en stock.  
- Les **Vaches à lait** doivent être maintenues, avec une optimisation progressive du prix.  
- Les **Dilemmes** méritent des tests de visibilité ou de promotion.  
- Les **Poids morts** peuvent être soldés, dépriorisés ou retirés du catalogue.
"""

display(Markdown(message))


### Lecture métier — BCG

La matrice BCG permet de distinguer les produits à fort potentiel des produits à faible contribution.  
Dans cette analyse, on identifie **513 produits Étoiles** et **417 produits Vaches à lait**.

- Les **Étoiles** doivent être mises en avant et sécurisées en stock.  
- Les **Vaches à lait** doivent être maintenues, avec une optimisation progressive du prix.  
- Les **Dilemmes** méritent des tests de visibilité ou de promotion.  
- Les **Poids morts** peuvent être soldés, dépriorisés ou retirés du catalogue.


## 12. Analyse de la saisonnalité

L'objectif est de comprendre à quel moment le catalogue vend le plus.

On analyse :

- le chiffre d'affaires par mois ;
- le chiffre d'affaires par semaine ;
- le chiffre d'affaires par jour de semaine ;
- une heatmap mois x jour de semaine.

In [33]:

# Variables temporelles
sales_st3["month"] = sales_st3["InvoiceDate"].dt.to_period("M").astype(str)
sales_st3["week"] = sales_st3["InvoiceDate"].dt.to_period("W").apply(lambda r: r.start_time)
sales_st3["weekday"] = sales_st3["InvoiceDate"].dt.day_name()

weekday_map = {
    "Monday": "Lundi",
    "Tuesday": "Mardi",
    "Wednesday": "Mercredi",
    "Thursday": "Jeudi",
    "Friday": "Vendredi",
    "Saturday": "Samedi",
    "Sunday": "Dimanche"
}

sales_st3["weekday_fr"] = sales_st3["weekday"].map(weekday_map)

# CA mensuel
monthly_revenue = (
    sales_st3.groupby("month", as_index=False)["TotalRevenue"]
    .sum()
    .rename(columns={"TotalRevenue": "monthly_revenue"})
)

# CA hebdomadaire
weekly_revenue = (
    sales_st3.groupby("week", as_index=False)["TotalRevenue"]
    .sum()
    .rename(columns={"TotalRevenue": "weekly_revenue"})
)

# CA par jour de semaine
weekday_order = ["Lundi", "Mardi", "Mercredi", "Jeudi", "Vendredi", "Samedi", "Dimanche"]
weekday_revenue = (
    sales_st3.groupby("weekday_fr", as_index=False)["TotalRevenue"]
    .sum()
    .rename(columns={"TotalRevenue": "weekday_revenue"})
)
weekday_revenue["weekday_fr"] = pd.Categorical(weekday_revenue["weekday_fr"], categories=weekday_order, ordered=True)
weekday_revenue = weekday_revenue.sort_values("weekday_fr")

print("CA mensuel :")
display(monthly_revenue)

CA mensuel :


,month,monthly_revenue
0,2010-12,"567,945.04"
1,2011-01,"434,350.28"
2,2011-02,"376,548.66"
3,2011-03,"499,042.58"
4,2011-04,"386,931.44"
5,2011-05,"546,022.31"
6,2011-06,"492,468.29"
7,2011-07,"497,990.06"
8,2011-08,"508,421.58"
9,2011-09,"737,847.72"


In [34]:

fig = px.line(
    monthly_revenue,
    x="month",
    y="monthly_revenue",
    markers=True,
    title="Évolution mensuelle du chiffre d'affaires",
    labels={"month": "Mois", "monthly_revenue": "Chiffre d'affaires"}
)

fig.update_layout(height=500)
fig.show()

In [35]:

fig = px.line(
    weekly_revenue,
    x="week",
    y="weekly_revenue",
    title="Évolution hebdomadaire du chiffre d'affaires",
    labels={"week": "Semaine", "weekly_revenue": "Chiffre d'affaires"}
)

fig.update_layout(height=500)
fig.show()

In [36]:

fig = px.bar(
    weekday_revenue,
    x="weekday_fr",
    y="weekday_revenue",
    title="Chiffre d'affaires par jour de semaine",
    labels={"weekday_fr": "Jour", "weekday_revenue": "Chiffre d'affaires"}
)

fig.update_layout(height=500)
fig.show()

In [37]:

# Heatmap mois x jour de semaine
heatmap_data = (
    sales_st3.groupby(["month", "weekday_fr"], as_index=False)["TotalRevenue"]
    .sum()
    .rename(columns={"TotalRevenue": "revenue"})
)

heatmap_pivot = heatmap_data.pivot(index="month", columns="weekday_fr", values="revenue")
heatmap_pivot = heatmap_pivot.reindex(columns=weekday_order)

fig = px.imshow(
    heatmap_pivot,
    aspect="auto",
    title="Heatmap saisonnalité — CA par mois et jour de semaine",
    labels={"x": "Jour de semaine", "y": "Mois", "color": "CA"}
)

fig.update_layout(height=650)
fig.show()

In [38]:

best_month_row = monthly_revenue.loc[monthly_revenue["monthly_revenue"].idxmax()]
weak_month_row = monthly_revenue.loc[monthly_revenue["monthly_revenue"].idxmin()]

message = f"""
### Lecture métier — Saisonnalité

Le mois le plus fort est **{best_month_row['month']}**, avec un chiffre d'affaires d'environ **{best_month_row['monthly_revenue']:,.0f}**.  
Le mois le plus faible est **{weak_month_row['month']}**, avec environ **{weak_month_row['monthly_revenue']:,.0f}**.

Cette information est utile pour planifier les stocks, les campagnes marketing et les promotions.  
Une promotion lancée au mauvais moment peut réduire la marge sans créer beaucoup de volume supplémentaire.
"""

display(Markdown(message))


### Lecture métier — Saisonnalité

Le mois le plus fort est **2011-11**, avec un chiffre d'affaires d'environ **1,073,123**.  
Le mois le plus faible est **2011-12**, avec environ **327,881**.

Cette information est utile pour planifier les stocks, les campagnes marketing et les promotions.  
Une promotion lancée au mauvais moment peut réduire la marge sans créer beaucoup de volume supplémentaire.


## 13. Décomposition STL indicative

La décomposition STL sépare une série temporelle en trois éléments :

- **tendance** : orientation générale du chiffre d'affaires ;
- **saisonnalité** : cycles réguliers ;
- **résidu** : variations exceptionnelles.

Comme la base couvre environ 13 mois, cette décomposition reste **indicative**. Elle aide surtout à préparer la lecture dashboard.

In [39]:

try:
    from statsmodels.tsa.seasonal import STL

    weekly_series = weekly_revenue.copy()
    weekly_series = weekly_series.set_index("week").sort_index()
    weekly_series = weekly_series.asfreq("W-MON")
    weekly_series["weekly_revenue"] = weekly_series["weekly_revenue"].fillna(0)

    # Période indicative de 13 semaines pour détecter des cycles infra-annuels
    stl = STL(weekly_series["weekly_revenue"], period=13, robust=True)
    result = stl.fit()

    stl_df = weekly_series.copy()
    stl_df["trend"] = result.trend
    stl_df["seasonal"] = result.seasonal
    stl_df["resid"] = result.resid

    fig = make_subplots(
        rows=4,
        cols=1,
        shared_xaxes=True,
        subplot_titles=["CA hebdomadaire", "Tendance", "Saisonnalité", "Résidu"]
    )

    fig.add_trace(go.Scatter(x=stl_df.index, y=stl_df["weekly_revenue"], mode="lines", name="CA"), row=1, col=1)
    fig.add_trace(go.Scatter(x=stl_df.index, y=stl_df["trend"], mode="lines", name="Tendance"), row=2, col=1)
    fig.add_trace(go.Scatter(x=stl_df.index, y=stl_df["seasonal"], mode="lines", name="Saisonnalité"), row=3, col=1)
    fig.add_trace(go.Scatter(x=stl_df.index, y=stl_df["resid"], mode="lines", name="Résidu"), row=4, col=1)

    fig.update_layout(height=900, title="Décomposition STL indicative du CA hebdomadaire")
    fig.show()

except Exception as e:
    print("STL non exécutée. Raison :", e)
    print("Si nécessaire, installer statsmodels : pip install statsmodels")

## 14. Analyse de l'élasticité prix

L'élasticité prix mesure la sensibilité des quantités vendues à une variation de prix.

Formule utilisée :

$$Elasticité = \frac{Variation\ \%\ de\ la\ quantité}{Variation\ \%\ du\ prix}$$

Interprétation simple :

- entre **0 et -1** : produit plutôt inélastique, une hausse de prix modérée peut être testée ;
- inférieur à **-1** : produit élastique, attention aux hausses de prix ;
- supérieur à **0** : comportement atypique ou effet produit complémentaire.

Pour rester robuste, on calcule l'élasticité sur les **50 produits les plus vendus**.

In [40]:
# Sélection des 50 produits les plus vendus
TOP_N_ELASTICITY = 50

top50_products = (
    product_perf
    .sort_values("quantity_sold", ascending=False)
    .head(TOP_N_ELASTICITY)["StockCode"]
)

# Agrégation mensuelle par produit
monthly_product = (
    sales_st3[sales_st3["StockCode"].isin(top50_products)]
    .groupby(["StockCode", "month"], as_index=False)
    .agg(
        monthly_qty=("Quantity", "sum"),
        monthly_revenue=("TotalRevenue", "sum")
    )
)

# Calcul du prix moyen mensuel
monthly_product["avg_monthly_price"] = (
    monthly_product["monthly_revenue"] / monthly_product["monthly_qty"]
)

monthly_product = monthly_product.sort_values(["StockCode", "month"])

# Variation du prix moyen et de la quantité vendue mois par mois
monthly_product["price_pct_change"] = (
    monthly_product.groupby("StockCode")["avg_monthly_price"].pct_change()
)

monthly_product["qty_pct_change"] = (
    monthly_product.groupby("StockCode")["monthly_qty"].pct_change()
)

# Calcul de l'élasticité observée
monthly_product["elasticity_obs"] = (
    monthly_product["qty_pct_change"] / monthly_product["price_pct_change"]
)

# Nettoyage des valeurs infinies
monthly_product = monthly_product.replace([np.inf, -np.inf], np.nan)

# On garde seulement les observations exploitables
elasticity_observations = monthly_product[
    (monthly_product["price_pct_change"].abs() >= 0.02) &
    (monthly_product["elasticity_obs"].abs() < 20)
].copy()

# Calcul de l'élasticité médiane par produit
elasticity = (
    elasticity_observations
    .groupby("StockCode", as_index=False)
    .agg(
        elasticity=("elasticity_obs", "median"),
        nb_elasticity_points=("elasticity_obs", "count")
    )
)

# Sécurité : si la cellule est relancée, on supprime les anciennes colonnes d'élasticité
cols_to_drop = [
    "elasticity",
    "elasticity_x",
    "elasticity_y",
    "nb_elasticity_points",
    "nb_elasticity_points_x",
    "nb_elasticity_points_y",
    "elasticity_profile"
]

product_perf = product_perf.drop(
    columns=[col for col in cols_to_drop if col in product_perf.columns],
    errors="ignore"
)

# Fusion propre avec la table produit
product_perf = product_perf.merge(elasticity, on="StockCode", how="left")

# Si aucune élasticité n'a pu être calculée, on crée quand même les colonnes
if "elasticity" not in product_perf.columns:
    product_perf["elasticity"] = np.nan

if "nb_elasticity_points" not in product_perf.columns:
    product_perf["nb_elasticity_points"] = 0

# Fonction de classification de l'élasticité
def classify_elasticity(value):
    if pd.isna(value):
        return "Non calculable"
    if -1 <= value < 0:
        return "Inélastique"
    if value < -1:
        return "Élastique"
    if value > 0:
        return "Atypique / complémentaire"
    return "Stable"

# Application du profil d'élasticité
product_perf["elasticity_profile"] = product_perf["elasticity"].apply(classify_elasticity)

# Aperçu des résultats
elasticity_preview = (
    product_perf[
        [
            "StockCode",
            "description",
            "quantity_sold",
            "avg_unit_price",
            "elasticity",
            "elasticity_profile",
            "nb_elasticity_points"
        ]
    ]
    .sort_values("quantity_sold", ascending=False)
    .head(20)
)

display(elasticity_preview)

,StockCode,description,quantity_sold,avg_unit_price,elasticity,elasticity_profile,nb_elasticity_points
3,85099B,JUMBO BAG RED RETROSPOT,"22,390.00",2.49,-1.32,Élastique,4.00
2,85123A,CREAM HANGING HEART T-LIGHT HOLDER,"21,629.50",3.12,-3.22,Élastique,4.00
5,84879,ASSORTED COLOUR BIRD ORNAMENT,"21,599.00",1.72,-2.24,Élastique,3.00
90,21212,PACK OF 72 RETROSPOT CAKE CASES,"20,153.50",0.76,-0.29,Inélastique,6.00
40,22197,POPCORN HOLDER,"19,262.50",1.04,3.67,Atypique / complémentaire,7.00
11,20725,LUNCH BAG RED RETROSPOT,"14,359.00",2.13,-0.61,Inélastique,10.00
30,22178,VICTORIAN GLASS HANGING T-LIGHT,"13,623.50",1.64,0.62,Atypique / complémentaire,7.00
482,84077,WORLD WAR 2 GLIDERS ASSTD DESIGNS,"13,543.50",0.32,9.31,Atypique / complémentaire,7.00
44,84946,ANTIQUE SILVER T-LIGHT GLASS,"12,741.50",1.53,-1.79,Élastique,12.00
1,47566,PARTY BUNTING,"12,613.00",5.45,-6.89,Élastique,10.00


In [41]:

elasticity_plot = product_perf.dropna(subset=["elasticity"]).copy()
elasticity_plot = elasticity_plot.sort_values("elasticity").head(30)
elasticity_plot["description_short"] = elasticity_plot["description"].str.slice(0, 55)

fig = px.bar(
    elasticity_plot,
    x="elasticity",
    y="description_short",
    orientation="h",
    title="Produits les plus sensibles au prix — élasticité estimée",
    labels={"elasticity": "Élasticité prix", "description_short": "Produit"},
    hover_data=["StockCode", "quantity_sold", "avg_unit_price", "elasticity_profile"]
)

fig.add_vline(x=-1, line_dash="dash", annotation_text="Seuil -1")
fig.add_vline(x=0, line_dash="dash", annotation_text="Seuil 0")
fig.update_layout(height=700)
fig.show()

### Lecture métier

L'élasticité n'est pas calculable pour tous les produits, car il faut observer une variation réelle du prix dans le temps.  
Quand elle est disponible, elle donne une première indication pour le pricing dynamique.

Il faut l'utiliser comme une aide à la décision, pas comme une vérité absolue.

## 15. Recommandations pricing

On combine maintenant plusieurs signaux :

- classe ABC ;
- quadrant BCG ;
- taux de retour ;
- élasticité prix ;
- mois de pic de ventes.

L'objectif est de produire une table simple et exploitable dans le dashboard.

In [42]:

# Mois de pic de vente par produit
product_month_revenue = (
    sales_st3.groupby(["StockCode", "month"], as_index=False)["TotalRevenue"]
    .sum()
    .rename(columns={"TotalRevenue": "month_revenue"})
)

peak_month = (
    product_month_revenue.sort_values(["StockCode", "month_revenue"], ascending=[True, False])
    .drop_duplicates("StockCode")
    .rename(columns={"month": "peak_month", "month_revenue": "peak_month_revenue"})
)

product_perf = product_perf.merge(
    peak_month[["StockCode", "peak_month", "peak_month_revenue"]],
    on="StockCode",
    how="left"
)

# Règles de recommandation simples et explicables
def pricing_recommendation(row):
    if row["return_rate_qty"] >= 0.20 and row["quantity_sold"] >= 20:
        return "Vérifier qualité / description avant promotion"

    if row["abc_class"] == "A" and row["elasticity_profile"] == "Inélastique":
        return "Tester une hausse de prix de 3 à 5 %"

    if row["bcg_quadrant"] == "Étoile":
        return "Mettre en avant et sécuriser le stock"

    if row["bcg_quadrant"] == "Dilemme":
        return "Tester visibilité ou promotion courte"

    if row["abc_class"] == "C" and row["bcg_quadrant"] == "Poids mort":
        return "Déprioriser, solder ou sortir du catalogue"

    if row["abc_class"] == "B":
        return "Maintenir et suivre la saisonnalité"

    return "Surveiller sans action urgente"

product_perf["pricing_action"] = product_perf.apply(pricing_recommendation, axis=1)

# Prix suggéré : uniquement pour les produits où on recommande un test de hausse
product_perf["suggested_price"] = np.where(
    product_perf["pricing_action"] == "Tester une hausse de prix de 3 à 5 %",
    product_perf["avg_unit_price"] * 1.05,
    product_perf["avg_unit_price"]
)

# Gain potentiel simple si hausse de 5 %, sans modélisation avancée de demande
product_perf["estimated_gain_5pct"] = np.where(
    product_perf["pricing_action"] == "Tester une hausse de prix de 3 à 5 %",
    product_perf["total_revenue"] * 0.05,
    0
)

recommendations_st3 = product_perf[
    [
        "StockCode", "description", "abc_class", "bcg_quadrant", "quantity_sold",
        "total_revenue", "avg_unit_price", "suggested_price", "elasticity",
        "elasticity_profile", "return_rate_qty", "peak_month", "pricing_action",
        "estimated_gain_5pct"
    ]
].copy()

# On affiche les recommandations les plus importantes par CA
recommendations_display = recommendations_st3.sort_values("total_revenue", ascending=False).head(30)
display(recommendations_display)

,StockCode,description,abc_class,bcg_quadrant,quantity_sold,total_revenue,avg_unit_price,suggested_price,elasticity,elasticity_profile,return_rate_qty,peak_month,pricing_action,estimated_gain_5pct
0,22423,REGENCY CAKESTAND 3 TIER,A,Vache à lait,"11,282.50","95,134.62",8.44,8.44,NaN,Non calculable,0.08,2010-12,Surveiller sans action urgente,0.00
1,47566,PARTY BUNTING,A,Étoile,"12,613.00","66,027.24",5.45,5.45,-6.89,Élastique,0.02,2011-05,Mettre en avant et sécuriser le stock,0.00
2,85123A,CREAM HANGING HEART T-LIGHT HOLDER,A,Vache à lait,"21,629.50","61,307.19",3.12,3.12,-3.22,Élastique,0.12,2011-05,Surveiller sans action urgente,0.00
3,85099B,JUMBO BAG RED RETROSPOT,A,Étoile,"22,390.00","47,772.21",2.49,2.49,-1.32,Élastique,0.05,2011-11,Mettre en avant et sécuriser le stock,0.00
4,22086,PAPER CHAIN KIT 50'S CHRISTMAS,A,Étoile,"11,493.50","37,507.46",3.36,3.36,-1.55,Élastique,0.04,2011-11,Mettre en avant et sécuriser le stock,0.00
5,84879,ASSORTED COLOUR BIRD ORNAMENT,A,Étoile,"21,599.00","36,728.65",1.72,1.72,-2.24,Élastique,0.00,2011-11,Mettre en avant et sécuriser le stock,0.00
6,23298,SPOTTY BUNTING,A,Étoile,"6,682.50","34,205.38",5.28,5.28,NaN,Non calculable,0.02,2011-05,Mettre en avant et sécuriser le stock,0.00
7,79321,CHILLI LIGHTS,A,Étoile,"6,250.50","33,013.13",6.01,6.01,NaN,Non calculable,0.01,2011-11,Mettre en avant et sécuriser le stock,0.00
8,22960,JAM MAKING SET WITH JARS,A,Vache à lait,"7,114.00","30,796.14",5.08,5.08,NaN,Non calculable,0.03,2011-03,Surveiller sans action urgente,0.00
9,22720,SET OF 3 CAKE TINS PANTRY DESIGN,A,Étoile,"5,638.50","28,418.85",5.47,5.47,NaN,Non calculable,0.03,2011-03,Mettre en avant et sécuriser le stock,0.00


In [43]:

action_summary = recommendations_st3.groupby("pricing_action", as_index=False).agg(
    nb_skus=("StockCode", "nunique"),
    total_revenue=("total_revenue", "sum"),
    estimated_gain_5pct=("estimated_gain_5pct", "sum")
).sort_values("total_revenue", ascending=False)

display(action_summary)

fig = px.bar(
    action_summary,
    x="pricing_action",
    y="total_revenue",
    title="Chiffre d'affaires couvert par type de recommandation pricing",
    labels={"pricing_action": "Action recommandée", "total_revenue": "Chiffre d'affaires"},
    hover_data=["nb_skus", "estimated_gain_5pct"]
)

fig.update_layout(height=600, xaxis_tickangle=-35)
fig.show()

,pricing_action,nb_skus,total_revenue,estimated_gain_5pct
2,Mettre en avant et sécuriser le stock,492,"3,506,301.39",0.00
3,Surveiller sans action urgente,405,"2,077,608.50",0.00
5,Tester visibilité ou promotion courte,1152,"659,729.05",0.00
1,Maintenir et suivre la saisonnalité,497,"541,126.43",0.00
0,"Déprioriser, solder ou sortir du catalogue",1269,"204,442.35",0.00
6,Vérifier qualité / description avant promotion,93,"164,078.50",0.00
4,Tester une hausse de prix de 3 à 5 %,7,"96,344.24","4,817.21"


In [44]:

gain_total = recommendations_st3["estimated_gain_5pct"].sum()
nb_price_increase = (recommendations_st3["pricing_action"] == "Tester une hausse de prix de 3 à 5 %").sum()

message = f"""
### Lecture métier — Recommandations pricing

Le notebook identifie **{nb_price_increase} produits** pour lesquels une hausse de prix modérée peut être testée.  
Le gain potentiel théorique associé à une hausse de 5 % est d'environ **{gain_total:,.0f}**.

Cette estimation doit être utilisée avec prudence : elle donne un ordre de grandeur, mais une vraie décision pricing doit aussi tenir compte de la concurrence, de la marge, du stock et du risque de baisse de volume.
"""

display(Markdown(message))


### Lecture métier — Recommandations pricing

Le notebook identifie **7 produits** pour lesquels une hausse de prix modérée peut être testée.  
Le gain potentiel théorique associé à une hausse de 5 % est d'environ **4,817**.

Cette estimation doit être utilisée avec prudence : elle donne un ordre de grandeur, mais une vraie décision pricing doit aussi tenir compte de la concurrence, de la marge, du stock et du risque de baisse de volume.


In [45]:
# ------------------------------------------------------------
# 1. Catégorisation produit estimée à partir du libellé
# ------------------------------------------------------------

def infer_product_category(description):
    """
    Catégorie estimée à partir du libellé produit.
    Le dataset UCI ne contient pas de vraie colonne catégorie.
    Cette règle permet d'avoir une analyse catégorie explicable.
    """
    text = str(description).upper()

    if any(k in text for k in ["CHRISTMAS", "XMAS", "SANTA", "NOEL", "ADVENT"]):
        return "Christmas"
    elif any(k in text for k in ["HEART", "HOME", "DECORATION", "HANGING", "SIGN", "WALL", "CANDLE"]):
        return "Home Decor"
    elif any(k in text for k in ["CAKE", "PLATE", "CUP", "MUG", "BOWL", "TEA", "KITCHEN", "SPOON"]):
        return "Kitchen & Tableware"
    elif any(k in text for k in ["CARD", "PAPER", "NOTEBOOK", "STICKER", "WRAP", "TAG"]):
        return "Stationery"
    elif any(k in text for k in ["BAG", "BOX", "PACK", "JUMBO", "LUNCH BAG"]):
        return "Bags & Packaging"
    elif any(k in text for k in ["TOY", "DOLL", "GAME", "PUZZLE", "CHILD", "BABY"]):
        return "Toys & Kids"
    elif any(k in text for k in ["NECKLACE", "BRACELET", "RING", "EARRING", "CHARM"]):
        return "Accessories"
    elif any(k in text for k in ["CUSHION", "BLANKET", "TOWEL", "APRON", "FABRIC"]):
        return "Textile"
    else:
        return "Other"


# Application sur les ventes
sales_st3["product_category"] = sales_st3["Description"].apply(infer_product_category)

# Application sur les retours si possible
if "Description" in returns_st3.columns:
    returns_st3["product_category"] = returns_st3["Description"].apply(infer_product_category)
else:
    returns_st3["product_category"] = "Other"

# Mapping SKU -> catégorie
product_category_map = (
    sales_st3.sort_values("InvoiceDate")
    .drop_duplicates("StockCode", keep="last")[["StockCode", "product_category"]]
)

# Ajout de la catégorie dans product_perf sans supprimer les anciennes colonnes
if "product_category" not in product_perf.columns:
    product_perf = product_perf.merge(product_category_map, on="StockCode", how="left")

product_perf["product_category"] = product_perf["product_category"].fillna("Other")

category_summary_st3 = (
    product_perf.groupby("product_category", as_index=False)
    .agg(
        nb_skus=("StockCode", "nunique"),
        total_revenue=("total_revenue", "sum"),
        quantity_sold=("quantity_sold", "sum"),
        avg_return_rate=("return_rate_qty", "mean")
    )
    .sort_values("total_revenue", ascending=False)
)

display(Markdown("### Synthèse par catégorie estimée"))
display(category_summary_st3)


# ------------------------------------------------------------
# 2. Mart mensuel produit : 1 ligne = 1 SKU × mois
# ------------------------------------------------------------

sales_st3["month"] = sales_st3["InvoiceDate"].dt.to_period("M").astype(str)

monthly_sku_sales = (
    sales_st3.groupby(["month", "StockCode", "product_category"], as_index=False)
    .agg(
        monthly_quantity=("Quantity", "sum"),
        monthly_revenue=("TotalRevenue", "sum"),
        monthly_orders=("InvoiceNo", "nunique"),
        monthly_customers=("CustomerID", "nunique"),
        avg_monthly_unit_price=("UnitPrice", "mean"),
        description=("Description", "last")
    )
)

returns_st3["InvoiceDate"] = pd.to_datetime(returns_st3["InvoiceDate"], errors="coerce")
returns_st3["month"] = returns_st3["InvoiceDate"].dt.to_period("M").astype(str)

monthly_sku_returns = (
    returns_st3.groupby(["month", "StockCode"], as_index=False)
    .agg(
        monthly_return_qty=("Quantity", lambda x: x.abs().sum()),
        monthly_return_value=("TotalRevenue", lambda x: x.abs().sum()),
        monthly_return_invoices=("InvoiceNo", "nunique")
    )
)

mart_product_perf_monthly_st3 = monthly_sku_sales.merge(
    monthly_sku_returns,
    on=["month", "StockCode"],
    how="left"
)

for col in ["monthly_return_qty", "monthly_return_value", "monthly_return_invoices"]:
    mart_product_perf_monthly_st3[col] = mart_product_perf_monthly_st3[col].fillna(0)

mart_product_perf_monthly_st3["monthly_return_rate"] = (
    mart_product_perf_monthly_st3["monthly_return_qty"] /
    mart_product_perf_monthly_st3["monthly_quantity"]
).replace([np.inf, -np.inf], np.nan).fillna(0)

monthly_total_revenue = (
    mart_product_perf_monthly_st3.groupby("month", as_index=False)["monthly_revenue"]
    .sum()
    .rename(columns={"monthly_revenue": "total_month_revenue"})
)

mart_product_perf_monthly_st3 = mart_product_perf_monthly_st3.merge(
    monthly_total_revenue,
    on="month",
    how="left"
)

mart_product_perf_monthly_st3["monthly_market_share_pct"] = (
    mart_product_perf_monthly_st3["monthly_revenue"] /
    mart_product_perf_monthly_st3["total_month_revenue"] * 100
)

# Élasticité mensuelle observée par SKU
mart_product_perf_monthly_st3 = mart_product_perf_monthly_st3.sort_values(["StockCode", "month"])

mart_product_perf_monthly_st3["price_pct_change"] = (
    mart_product_perf_monthly_st3.groupby("StockCode")["avg_monthly_unit_price"].pct_change()
)

mart_product_perf_monthly_st3["qty_pct_change"] = (
    mart_product_perf_monthly_st3.groupby("StockCode")["monthly_quantity"].pct_change()
)

mart_product_perf_monthly_st3["monthly_elasticity"] = (
    mart_product_perf_monthly_st3["qty_pct_change"] /
    mart_product_perf_monthly_st3["price_pct_change"]
).replace([np.inf, -np.inf], np.nan)

# Ajout des dimensions déjà calculées dans product_perf
monthly_dims_cols = [
    "StockCode", "abc_class", "bcg_quadrant", "elasticity",
    "elasticity_profile", "pricing_action", "suggested_price",
    "estimated_gain_5pct", "return_rate_qty"
]

monthly_dims_cols = [col for col in monthly_dims_cols if col in product_perf.columns]

mart_product_perf_monthly_st3 = mart_product_perf_monthly_st3.merge(
    product_perf[monthly_dims_cols],
    on="StockCode",
    how="left"
)

display(Markdown("### Mart mensuel SKU × mois"))
display(mart_product_perf_monthly_st3.head(20))
print("Nombre de lignes mart mensuel :", len(mart_product_perf_monthly_st3))


# ------------------------------------------------------------
# 3. Heatmap saisonnalité : mois × catégorie
# ------------------------------------------------------------

seasonality_category_st3 = (
    sales_st3.groupby(["month", "product_category"], as_index=False)
    .agg(
        monthly_revenue=("TotalRevenue", "sum"),
        monthly_quantity=("Quantity", "sum"),
        nb_orders=("InvoiceNo", "nunique")
    )
)

category_monthly_avg = (
    seasonality_category_st3.groupby("product_category", as_index=False)["monthly_revenue"]
    .mean()
    .rename(columns={"monthly_revenue": "avg_monthly_revenue_category"})
)

seasonality_category_st3 = seasonality_category_st3.merge(
    category_monthly_avg,
    on="product_category",
    how="left"
)

seasonality_category_st3["seasonality_index"] = (
    seasonality_category_st3["monthly_revenue"] /
    seasonality_category_st3["avg_monthly_revenue_category"]
).replace([np.inf, -np.inf], np.nan).fillna(0)

seasonality_category_st3["seasonality_status"] = np.select(
    [
        seasonality_category_st3["seasonality_index"] >= 1.2,
        seasonality_category_st3["seasonality_index"] <= 0.8
    ],
    ["Mois fort", "Mois faible"],
    default="Mois normal"
)

heatmap_category = seasonality_category_st3.pivot(
    index="product_category",
    columns="month",
    values="seasonality_index"
).fillna(0)

fig = px.imshow(
    heatmap_category,
    aspect="auto",
    title="Heatmap saisonnalité — Mois × catégorie",
    labels={"x": "Mois", "y": "Catégorie", "color": "Indice saisonnier"}
)

fig.update_layout(height=650)
fig.show()

display(Markdown("### Top pics saisonniers par catégorie"))
display(seasonality_category_st3.sort_values("seasonality_index", ascending=False).head(20))


# ------------------------------------------------------------
# 4. Courbe Pareto exportable
# ------------------------------------------------------------

pareto_curve_st3 = product_perf[
    [
        "StockCode", "description", "product_category", "rank_revenue",
        "total_revenue", "cumulative_revenue_pct", "abc_class"
    ]
].copy()

pareto_curve_st3 = pareto_curve_st3.sort_values("rank_revenue").reset_index(drop=True)
pareto_curve_st3["sku_count"] = 1
pareto_curve_st3["cumulative_sku_count"] = pareto_curve_st3["sku_count"].cumsum()
pareto_curve_st3["cumulative_sku_pct"] = (
    pareto_curve_st3["cumulative_sku_count"] /
    pareto_curve_st3["sku_count"].sum() * 100
)

fig = px.line(
    pareto_curve_st3,
    x="cumulative_sku_pct",
    y="cumulative_revenue_pct",
    title="Courbe Pareto — % cumulé SKUs vs % cumulé CA",
    labels={
        "cumulative_sku_pct": "% cumulé des SKUs",
        "cumulative_revenue_pct": "% cumulé du chiffre d'affaires"
    },
    hover_data=["StockCode", "description", "product_category", "abc_class", "total_revenue"]
)

fig.add_hline(y=80, line_dash="dash", annotation_text="80 % CA")
fig.add_hline(y=95, line_dash="dash", annotation_text="95 % CA")
fig.update_layout(height=500)
fig.show()


# ------------------------------------------------------------
# 5. Décomposition STL exportable : total + top catégories
# ------------------------------------------------------------

sales_st3["week"] = sales_st3["InvoiceDate"].dt.to_period("W").apply(lambda r: r.start_time)

def build_stl_table(df, value_col="TotalRevenue", date_col="week", scope_name="Total catalogue"):
    """
    Décomposition STL hebdomadaire.
    Retourne une table exportable pour Tableau.
    """
    try:
        from statsmodels.tsa.seasonal import STL
    except ImportError:
        print("statsmodels n'est pas installé. Installer avec : pip install statsmodels")
        return pd.DataFrame()

    series = (
        df.groupby(date_col, as_index=False)[value_col]
        .sum()
        .rename(columns={value_col: "weekly_revenue"})
    )

    series = series.set_index(date_col).sort_index()
    series = series.asfreq("W-MON")
    series["weekly_revenue"] = series["weekly_revenue"].fillna(0)

    if len(series) < 30:
        return pd.DataFrame()

    result = STL(series["weekly_revenue"], period=13, robust=True).fit()

    out = series.reset_index()
    out["scope"] = scope_name
    out["trend"] = result.trend.values
    out["seasonal"] = result.seasonal.values
    out["resid"] = result.resid.values

    mean_revenue = out["weekly_revenue"].mean()

    out["seasonality_index"] = np.where(
        mean_revenue > 0,
        (out["trend"] + out["seasonal"]) / mean_revenue,
        np.nan
    )

    out["seasonality_status"] = np.select(
        [
            out["seasonality_index"] >= 1.2,
            out["seasonality_index"] <= 0.8
        ],
        ["Semaine forte", "Semaine faible"],
        default="Semaine normale"
    )

    return out


stl_tables = []

stl_total = build_stl_table(
    sales_st3,
    value_col="TotalRevenue",
    date_col="week",
    scope_name="Total catalogue"
)

if not stl_total.empty:
    stl_tables.append(stl_total)

top_categories = (
    sales_st3.groupby("product_category")["TotalRevenue"]
    .sum()
    .sort_values(ascending=False)
    .head(5)
    .index
    .tolist()
)

for category in top_categories:
    df_cat = sales_st3[sales_st3["product_category"] == category].copy()
    stl_cat = build_stl_table(
        df_cat,
        value_col="TotalRevenue",
        date_col="week",
        scope_name=category
    )

    if not stl_cat.empty:
        stl_tables.append(stl_cat)

if len(stl_tables) > 0:
    stl_decomposition_st3 = pd.concat(stl_tables, ignore_index=True)
else:
    stl_decomposition_st3 = pd.DataFrame()

display(Markdown("### Décomposition STL exportable"))
display(stl_decomposition_st3.head(20))

# Visualisation rapide du total catalogue
if not stl_decomposition_st3.empty:
    stl_total_plot = stl_decomposition_st3[
        stl_decomposition_st3["scope"] == "Total catalogue"
    ].copy()

    if not stl_total_plot.empty:
        fig = make_subplots(
            rows=4,
            cols=1,
            shared_xaxes=True,
            subplot_titles=["CA hebdomadaire", "Tendance", "Saisonnalité", "Résidu"]
        )

        fig.add_trace(go.Scatter(x=stl_total_plot["week"], y=stl_total_plot["weekly_revenue"], mode="lines", name="CA"), row=1, col=1)
        fig.add_trace(go.Scatter(x=stl_total_plot["week"], y=stl_total_plot["trend"], mode="lines", name="Tendance"), row=2, col=1)
        fig.add_trace(go.Scatter(x=stl_total_plot["week"], y=stl_total_plot["seasonal"], mode="lines", name="Saisonnalité"), row=3, col=1)
        fig.add_trace(go.Scatter(x=stl_total_plot["week"], y=stl_total_plot["resid"], mode="lines", name="Résidu"), row=4, col=1)

        fig.update_layout(height=850, title="Décomposition STL — Total catalogue")
        fig.show()


# ------------------------------------------------------------
# 6. Élasticité par catégorie
# ------------------------------------------------------------

elasticity_category_source = mart_product_perf_monthly_st3[
    mart_product_perf_monthly_st3["monthly_elasticity"].notna()
].copy()

elasticity_category_source = elasticity_category_source[
    elasticity_category_source["monthly_elasticity"].between(-20, 20)
].copy()

if len(elasticity_category_source) > 0:
    elasticity_category_st3 = (
        elasticity_category_source.groupby("product_category", as_index=False)
        .agg(
            median_elasticity=("monthly_elasticity", "median"),
            avg_elasticity=("monthly_elasticity", "mean"),
            nb_observations=("monthly_elasticity", "count"),
            total_revenue=("monthly_revenue", "sum"),
            total_quantity=("monthly_quantity", "sum")
        )
    )
else:
    elasticity_category_st3 = pd.DataFrame(
        columns=[
            "product_category", "median_elasticity", "avg_elasticity",
            "nb_observations", "total_revenue", "total_quantity"
        ]
    )

def classify_category_elasticity(value):
    if pd.isna(value):
        return "Non calculable"
    if -1 <= value < 0:
        return "Inélastique"
    elif value < -1:
        return "Élastique"
    elif value > 0:
        return "Atypique / complémentaire"
    else:
        return "Stable"

elasticity_category_st3["elasticity_profile"] = elasticity_category_st3["median_elasticity"].apply(
    classify_category_elasticity
)

elasticity_category_st3 = elasticity_category_st3.sort_values("total_revenue", ascending=False)

if not elasticity_category_st3.empty:
    fig = px.bar(
        elasticity_category_st3,
        x="product_category",
        y="median_elasticity",
        color="elasticity_profile",
        title="Élasticité prix médiane par catégorie",
        labels={
            "product_category": "Catégorie",
            "median_elasticity": "Élasticité médiane",
            "elasticity_profile": "Profil"
        },
        hover_data=["nb_observations", "total_revenue", "total_quantity"]
    )

    fig.add_hline(y=-1, line_dash="dash", annotation_text="Seuil -1")
    fig.add_hline(y=0, line_dash="dash", annotation_text="Seuil 0")
    fig.update_layout(height=600, xaxis_tickangle=-35)
    fig.show()

display(Markdown("### Élasticité par catégorie"))
display(elasticity_category_st3)


# ------------------------------------------------------------
# 7. Fenêtres promotionnelles optimales
# ------------------------------------------------------------

strong_seasonality = seasonality_category_st3[
    seasonality_category_st3["seasonality_index"] >= 1.2
].copy()

promo_windows_st3 = (
    strong_seasonality.groupby("product_category", as_index=False)
    .agg(
        best_months=("month", lambda x: ", ".join(sorted(x.astype(str).unique()))),
        nb_strong_months=("month", "nunique"),
        revenue_in_peak_months=("monthly_revenue", "sum"),
        avg_seasonality_index=("seasonality_index", "mean"),
        max_seasonality_index=("seasonality_index", "max")
    )
    .sort_values("revenue_in_peak_months", ascending=False)
)

promo_windows_st3["recommendation"] = np.where(
    promo_windows_st3["nb_strong_months"] >= 2,
    "Planifier promotions ciblées sur les mois forts",
    "Tester une campagne courte sur le mois fort"
)

top3_promo_windows_st3 = promo_windows_st3.head(3).copy()

display(Markdown("### Top 3 fenêtres promotionnelles optimales"))
display(top3_promo_windows_st3)


# ------------------------------------------------------------
# 8. Top 5 recommandations pricing chiffrées
# ------------------------------------------------------------

pricing_top5_recommendations_st3 = recommendations_st3[
    recommendations_st3["pricing_action"] == "Tester une hausse de prix de 3 à 5 %"
].copy()

pricing_top5_recommendations_st3 = pricing_top5_recommendations_st3[
    pricing_top5_recommendations_st3["return_rate_qty"] <= 0.20
].copy()

pricing_top5_recommendations_st3 = pricing_top5_recommendations_st3.sort_values(
    "estimated_gain_5pct",
    ascending=False
).head(5)

# Si moins de 5 produits respectent les critères, on complète avec les meilleurs produits en CA
if len(pricing_top5_recommendations_st3) < 5:
    fallback = recommendations_st3[
        ~recommendations_st3["StockCode"].isin(pricing_top5_recommendations_st3["StockCode"])
    ].sort_values("total_revenue", ascending=False).head(5 - len(pricing_top5_recommendations_st3))

    pricing_top5_recommendations_st3 = pd.concat(
        [pricing_top5_recommendations_st3, fallback],
        ignore_index=True
    )

pricing_top5_recommendations_st3["pricing_comment"] = (
    "Prix actuel : "
    + pricing_top5_recommendations_st3["avg_unit_price"].round(2).astype(str)
    + " | Prix suggéré : "
    + pricing_top5_recommendations_st3["suggested_price"].round(2).astype(str)
    + " | Gain estimé : "
    + pricing_top5_recommendations_st3["estimated_gain_5pct"].round(0).astype(str)
)

display(Markdown("### Top 5 recommandations pricing chiffrées"))
display(pricing_top5_recommendations_st3[
    [
        "StockCode", "description", "abc_class", "bcg_quadrant",
        "total_revenue", "avg_unit_price", "suggested_price",
        "elasticity", "elasticity_profile", "return_rate_qty",
        "estimated_gain_5pct", "pricing_action", "pricing_comment"
    ]
])

total_gain_top5 = pricing_top5_recommendations_st3["estimated_gain_5pct"].sum()

display(Markdown(f"""
### Lecture métier — Pricing

Les 5 recommandations pricing représentent un gain potentiel théorique d’environ **{total_gain_top5:,.0f}** avec une hausse simulée de 5 %.  

Ces recommandations doivent être considérées comme des tests contrôlés, car l’élasticité est observée sur l’historique et ne prouve pas une causalité parfaite.
"""))

### Synthèse par catégorie estimée

,product_category,nb_skus,total_revenue,quantity_sold,avg_return_rate
5,Other,1411,"2,599,694.54","1,129,965.00",0.07
3,Home Decor,850,"1,658,361.90","910,923.00",0.03
4,Kitchen & Tableware,321,"827,325.84","387,873.50",0.05
1,Bags & Packaging,266,"812,701.87","444,044.00",0.03
6,Stationery,426,"629,682.35","442,418.00",6.69
2,Christmas,163,"364,320.88","233,025.00",0.04
8,Toys & Kids,82,"196,059.92","64,630.00",0.02
0,Accessories,325,"98,419.77","61,161.00",0.02
7,Textile,71,"63,063.39","21,274.00",0.02


### Mart mensuel SKU × mois

,month,StockCode,product_category,monthly_quantity,monthly_revenue,monthly_orders,monthly_customers,avg_monthly_unit_price,description,monthly_return_qty,monthly_return_value,monthly_return_invoices,monthly_return_rate,total_month_revenue,monthly_market_share_pct,price_pct_change,qty_pct_change,monthly_elasticity,abc_class,bcg_quadrant,elasticity,elasticity_profile,pricing_action,suggested_price,estimated_gain_5pct,return_rate_qty
0,2010-12,10002,Other,224.00,211.46,30,15,1.20,INFLATABLE POLITICAL GLOBE,0.00,0.00,0.00,0.00,"567,945.04",0.04,NaN,NaN,NaN,C,Poids mort,NaN,Non calculable,"Déprioriser, solder ou sortir du catalogue",1.09,0.00,0.00
1,2011-01,10002,Other,217.00,186.82,21,18,0.96,INFLATABLE POLITICAL GLOBE,0.00,0.00,0.00,0.00,"434,350.28",0.04,-0.20,-0.03,0.16,C,Poids mort,NaN,Non calculable,"Déprioriser, solder ou sortir du catalogue",1.09,0.00,0.00
2,2011-02,10002,Other,52.00,45.76,7,5,1.07,INFLATABLE POLITICAL GLOBE,0.00,0.00,0.00,0.00,"376,548.66",0.01,0.11,-0.76,-6.66,C,Poids mort,NaN,Non calculable,"Déprioriser, solder ou sortir du catalogue",1.09,0.00,0.00
3,2011-03,10002,Other,28.00,27.70,8,5,1.14,INFLATABLE POLITICAL GLOBE,0.00,0.00,0.00,0.00,"499,042.58",0.01,0.06,-0.46,-7.11,C,Poids mort,NaN,Non calculable,"Déprioriser, solder ou sortir du catalogue",1.09,0.00,0.00
4,2011-04,10002,Other,64.00,54.40,5,5,0.85,INFLATABLE POLITICAL GLOBE,0.00,0.00,0.00,0.00,"386,931.44",0.01,-0.26,1.29,-5.02,C,Poids mort,NaN,Non calculable,"Déprioriser, solder ou sortir du catalogue",1.09,0.00,0.00
5,2011-02,10080,Other,2.00,1.70,1,1,0.85,GROOVY CACTUS INFLATABLE,0.00,0.00,0.00,0.00,"376,548.66",0.00,NaN,NaN,NaN,C,Dilemme,NaN,Non calculable,Tester visibilité ou promotion courte,0.41,0.00,0.00
6,2011-06,10080,Other,40.50,15.79,2,1,0.39,GROOVY CACTUS INFLATABLE,0.00,0.00,0.00,0.00,"492,468.29",0.00,-0.54,19.25,-35.57,C,Dilemme,NaN,Non calculable,Tester visibilité ou promotion courte,0.41,0.00,0.00
7,2011-07,10080,Other,24.00,9.36,2,2,0.39,GROOVY CACTUS INFLATABLE,0.00,0.00,0.00,0.00,"497,990.06",0.00,0.00,-0.41,NaN,C,Dilemme,NaN,Non calculable,Tester visibilité ou promotion courte,0.41,0.00,0.00
8,2011-08,10080,Other,60.00,23.40,4,4,0.39,GROOVY CACTUS INFLATABLE,0.00,0.00,0.00,0.00,"508,421.58",0.00,0.00,1.50,NaN,C,Dilemme,NaN,Non calculable,Tester visibilité ou promotion courte,0.41,0.00,0.00
9,2011-09,10080,Other,60.00,23.40,4,4,0.39,GROOVY CACTUS INFLATABLE,0.00,0.00,0.00,0.00,"737,847.72",0.00,0.00,0.00,NaN,C,Dilemme,NaN,Non calculable,Tester visibilité ou promotion courte,0.41,0.00,0.00


Nombre de lignes mart mensuel : 34092


...

### Top 3 fenêtres promotionnelles optimales

,product_category,best_months,nb_strong_months,revenue_in_peak_months,avg_seasonality_index,max_seasonality_index,recommendation
5,Other,"2011-09, 2011-10, 2011-11",3,"888,195.22",1.48,1.88,Planifier promotions ciblées sur les mois forts
3,Home Decor,"2011-09, 2011-10, 2011-11",3,"598,081.60",1.56,1.83,Planifier promotions ciblées sur les mois forts
2,Christmas,"2010-12, 2011-09, 2011-10, 2011-11",4,"303,947.98",2.70,4.35,Planifier promotions ciblées sur les mois forts


### Top 5 recommandations pricing chiffrées

,StockCode,description,abc_class,bcg_quadrant,total_revenue,avg_unit_price,suggested_price,elasticity,elasticity_profile,return_rate_qty,estimated_gain_5pct,pricing_action,pricing_comment
11,20725,LUNCH BAG RED RETROSPOT,A,Étoile,"27,816.06",2.13,2.24,-0.61,Inélastique,0.04,"1,390.80",Tester une hausse de prix de 3 à 5 %,Prix actuel : 2.13 | Prix suggéré : 2.24 | Gai...
39,22383,LUNCH BAG SUKI DESIGN,A,Étoile,"18,783.14",2.17,2.28,-0.72,Inélastique,0.01,939.16,Tester une hausse de prix de 3 à 5 %,Prix actuel : 2.17 | Prix suggéré : 2.28 | Gai...
90,21212,PACK OF 72 RETROSPOT CAKE CASES,A,Vache à lait,"13,416.78",0.76,0.80,-0.29,Inélastique,0.01,670.84,Tester une hausse de prix de 3 à 5 %,Prix actuel : 0.76 | Prix suggéré : 0.8 | Gain...
92,20724,RED RETROSPOT CHARLOTTE BAG,A,Étoile,"13,228.92",1.15,1.20,-0.53,Inélastique,0.02,661.45,Tester une hausse de prix de 3 à 5 %,Prix actuel : 1.15 | Prix suggéré : 1.2 | Gain...
188,21790,VINTAGE SNAP CARDS,A,Étoile,"8,153.76",1.02,1.07,-0.34,Inélastique,0.01,407.69,Tester une hausse de prix de 3 à 5 %,Prix actuel : 1.02 | Prix suggéré : 1.07 | Gai...



### Lecture métier — Pricing

Les 5 recommandations pricing représentent un gain potentiel théorique d’environ **4,070** avec une hausse simulée de 5 %.  

Ces recommandations doivent être considérées comme des tests contrôlés, car l’élasticité est observée sur l’historique et ne prouve pas une causalité parfaite.


## 16. Exports pour L8 Dashboard

On exporte les fichiers nécessaires pour construire la page Power BI / Tableau **ST3 — Catalogue Intelligence**.

Les fichiers générés dans `data/gold/` sont :

- `mart_product_perf_eda.csv` : table complète produit ;
- `top20_skus_st3.csv` : Top 20 produits ;
- `abc_summary_st3.csv` : synthèse ABC ;
- `bcg_summary_st3.csv` : synthèse BCG ;
- `seasonality_monthly_st3.csv` : saisonnalité mensuelle ;
- `seasonality_weekly_st3.csv` : saisonnalité hebdomadaire ;
- `pricing_recommendations_st3.csv` : recommandations pricing.

In [46]:
# ============================================================
# 16. Exports complets ST3 pour L8 Dashboard
# ============================================================

# Important :
# On conserve les anciens fichiers et les anciennes colonnes pour ne pas casser Tableau.
# On ajoute seulement des colonnes et des fichiers complémentaires.

product_export_cols = [
    "StockCode", "description", "product_category", "quantity_sold", "total_revenue",
    "nb_orders", "nb_customers", "avg_unit_price", "returned_qty", "return_value",
    "return_rate_qty", "market_share_pct", "rank_revenue", "cumulative_revenue_pct",
    "abc_class", "first_period", "second_period", "growth_pct", "bcg_quadrant",
    "elasticity", "elasticity_profile", "peak_month", "pricing_action",
    "suggested_price", "estimated_gain_5pct"
]

product_export_cols = [col for col in product_export_cols if col in product_perf.columns]

# Exports existants utilisés par ton dashboard actuel
product_perf[product_export_cols].to_csv(
    GOLD_DIR / "mart_product_perf_eda.csv",
    index=False,
    encoding="utf-8-sig"
)

top20_skus.to_csv(
    GOLD_DIR / "top20_skus_st3.csv",
    index=False,
    encoding="utf-8-sig"
)

abc_summary.to_csv(
    GOLD_DIR / "abc_summary_st3.csv",
    index=False,
    encoding="utf-8-sig"
)

bcg_summary.to_csv(
    GOLD_DIR / "bcg_summary_st3.csv",
    index=False,
    encoding="utf-8-sig"
)

monthly_revenue.to_csv(
    GOLD_DIR / "seasonality_monthly_st3.csv",
    index=False,
    encoding="utf-8-sig"
)

weekly_revenue.to_csv(
    GOLD_DIR / "seasonality_weekly_st3.csv",
    index=False,
    encoding="utf-8-sig"
)

recommendations_st3.to_csv(
    GOLD_DIR / "pricing_recommendations_st3.csv",
    index=False,
    encoding="utf-8-sig"
)

# Nouveaux exports complémentaires CDC
category_summary_st3.to_csv(
    GOLD_DIR / "category_summary_st3.csv",
    index=False,
    encoding="utf-8-sig"
)

mart_product_perf_monthly_st3.to_csv(
    GOLD_DIR / "mart_product_perf_monthly_st3.csv",
    index=False,
    encoding="utf-8-sig"
)

seasonality_category_st3.to_csv(
    GOLD_DIR / "seasonality_category_st3.csv",
    index=False,
    encoding="utf-8-sig"
)

pareto_curve_st3.to_csv(
    GOLD_DIR / "pareto_curve_st3.csv",
    index=False,
    encoding="utf-8-sig"
)

stl_decomposition_st3.to_csv(
    GOLD_DIR / "stl_decomposition_st3.csv",
    index=False,
    encoding="utf-8-sig"
)

elasticity_category_st3.to_csv(
    GOLD_DIR / "elasticity_category_st3.csv",
    index=False,
    encoding="utf-8-sig"
)

promo_windows_st3.to_csv(
    GOLD_DIR / "promo_windows_st3.csv",
    index=False,
    encoding="utf-8-sig"
)

top3_promo_windows_st3.to_csv(
    GOLD_DIR / "top3_promo_windows_st3.csv",
    index=False,
    encoding="utf-8-sig"
)

pricing_top5_recommendations_st3.to_csv(
    GOLD_DIR / "pricing_top5_recommendations_st3.csv",
    index=False,
    encoding="utf-8-sig"
)

exports = [
    "mart_product_perf_eda.csv",
    "top20_skus_st3.csv",
    "abc_summary_st3.csv",
    "bcg_summary_st3.csv",
    "seasonality_monthly_st3.csv",
    "seasonality_weekly_st3.csv",
    "pricing_recommendations_st3.csv",
    "category_summary_st3.csv",
    "mart_product_perf_monthly_st3.csv",
    "seasonality_category_st3.csv",
    "pareto_curve_st3.csv",
    "stl_decomposition_st3.csv",
    "elasticity_category_st3.csv",
    "promo_windows_st3.csv",
    "top3_promo_windows_st3.csv",
    "pricing_top5_recommendations_st3.csv"
]

print("Exports ST3 terminés dans :", GOLD_DIR)

for file in exports:
    path = GOLD_DIR / file
    print("-", file, "OK" if path.exists() else "MANQUANT")

Exports ST3 terminés dans : C:\Users\Olfa\OneDrive\Bureau\OneDrive\Documents\Downloads\ecommerce-analytics-project\data\gold
- mart_product_perf_eda.csv OK
- top20_skus_st3.csv OK
- abc_summary_st3.csv OK
- bcg_summary_st3.csv OK
- seasonality_monthly_st3.csv OK
- seasonality_weekly_st3.csv OK
- pricing_recommendations_st3.csv OK
- category_summary_st3.csv OK
- mart_product_perf_monthly_st3.csv OK
- seasonality_category_st3.csv OK
- pareto_curve_st3.csv OK
- stl_decomposition_st3.csv OK
- elasticity_category_st3.csv OK
- promo_windows_st3.csv OK
- top3_promo_windows_st3.csv OK
- pricing_top5_recommendations_st3.csv OK
